# Set up for automated input

The idea of this notebook is to check whether we can select ERA5 and SLR data point automatically from a provided bbox 

In [ ]:
import duckdb

import hvplot.pandas 
import hvplot.xarray

import numpy as np
import geopandas as gpd
import xarray as xr

import shapely

from dotenv import load_dotenv
import os 

import holoviews as hv
hv.extension('bokeh')

In [ ]:
# load CDS API key
load_dotenv()
cds_api_key = os.getenv('CDS-API-KEY')


In [ ]:
# start with selected bbox 
# BBOX = [81.2366, 8.5625, 81.2500, 8.5763] # dutch bay 
BBOX = [81.809224, 7.412626, 81.850069, 7.460627] # B-III
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = BBOX
PARQUET_PATH = '../data/gctr/*.parquet'

In [ ]:
gpd.GeoDataFrame(
    geometry=[shapely.box(*BBOX)], 
    crs='EPSG:4326'
).hvplot(
    geo=True, 
    tiles='EsriImagery', 
    width=600, 
    height=400, 
    fill_color=None, 
    line_color='red'
)

## Import Transects 
Inspect Parquet File with duckdb

In [ ]:
con = duckdb.connect()

In [ ]:
# Summmarize the min, max, avg, null per column 
con.sql(f"SUMMARIZE SELECT * FROM '{PARQUET_PATH}'").show()

In [ ]:
# # some usefull inspecting framework
# def inspect(path: str):
#     from pathlib import Path

#     con = duckdb.connect()
#     con.sql("INSTALL spatial; LOAD spatial;")  # harmless no-op if file has no geometry

#     print(f"=== {path} ===")
#     print(f"file size: {Path(path).stat().st_size / 1e6:.1f} MB\n")

#     print("--- schema ---")
#     schema = con.sql(f"DESCRIBE SELECT * FROM '{path}'").fetchall()
#     geom_cols = []
#     for name, col_type, *_ in schema:
#         short_type = col_type if len(col_type) < 60 else col_type[:57] + "..."
#         print(f"  {name:35s} {short_type}")
#         if col_type.startswith("GEOMETRY"):
#             geom_cols.append(name)

#     n = con.sql(f"SELECT COUNT(*) FROM '{path}'").fetchone()[0]
#     print(f"\nrow count: {n:,}\n")

#     print("--- summary stats (SUMMARIZE) ---")
#     summary = con.sql(f"SUMMARIZE SELECT * FROM '{path}'").df()
#     cols_to_show = ["column_name", "column_type", "min", "max", "null_percentage"]
#     print(summary[cols_to_show].to_string(index=False))

#     print("\n--- sample rows ---")
#     print(con.sql(f"SELECT * FROM '{path}' LIMIT 3").df().to_string())

#     if geom_cols:
#         print("\n--- geometry ---")
#         for col in geom_cols:
#             extent = con.sql(f"""
#                 SELECT ST_Extent_Agg({col}) FROM '{path}'
#             """).fetchone()[0]
#             gtypes = con.sql(f"""
#                 SELECT DISTINCT ST_GeometryType({col}) AS geom_type
#                 FROM '{path}' USING SAMPLE 10000
#             """).fetchall()
#             print(f"  {col}: extent={extent}, sample geom types={[g[0] for g in gtypes]}")

Filter the parquet file within bounding box

In [ ]:
df = con.sql(f"""
    SELECT transect_id, lon, lat, bearing, utm_epsg, country, ST_AsText(geometry) AS wkt
    FROM '{PARQUET_PATH}'
    WHERE lon BETWEEN {LON_MIN} AND {LON_MAX}
      AND lat BETWEEN {LAT_MIN} AND {LAT_MAX}
""").df()

In [ ]:
# convert into geopandas dataframe 
gdf = gpd.GeoDataFrame(
    df.drop(columns='wkt'),
    geometry=gpd.GeoSeries.from_wkt(df['wkt']), 
    crs='EPSG:4326'
)
print(f'length of the coastline data: {len(gdf)*100} m')
gdf.head()

In [ ]:
a = gdf.sort_values(by='transect_id').hvplot(
    geo=True, tiles="EsriImagery",
    hover_cols=["transect_id", 'bearing'], color='bearing', size=30,
)

b = gdf.hvplot.points(
    x='lon', 
    y='lat',
    geo=True,
)

c = gpd.GeoDataFrame(
    geometry=[shapely.geometry.box(*BBOX)], 
    crs='EPSG:4326', 
).hvplot(
    geo=True, 
    width=600, 
    height=400, 
).opts(fill_color=None, line_color='red')

## Select ERA5 closest points

In [ ]:
# load the ERA5 dataset

# Geo-chunked wave data (optimised for time-series at a single location)
geochunked_wav_url = "https://arco.datastores.ecmwf.int/cadl-arco-geo-003/arco/reanalysis_era5_single_levels/wav/geoChunked.zarr"
timechunked_wav_url = "https://arco.datastores.ecmwf.int/cadl-arco-geo-003/arco/reanalysis_era5_single_levels/wav/timeChunked.zarr"

# load the data lazily
ds = xr.open_zarr(
    geochunked_wav_url, 
    consolidated=True, 
    storage_options={
        "headers": {
            "Authorization": f"Bearer {os.getenv('CDS-API-KEY')}"
        }
    }
)

# inspect the variable 
ds

In [ ]:
# select middle transect 
transects_df = gdf.sort_values(by='transect_id').reset_index(drop=True)
transect = transects_df.iloc[len(transects_df)//2]

# clip the data within middle transect buffer 
buffer = 1 # 1 degree buffer around the bbox 

era5_buffer = ds.sel(longitude=slice(transect.lon - buffer, transect.lon + buffer), latitude=slice(transect.lat - buffer, transect.lat + buffer)).isel(time=0).compute()

# convert to dataframe 
era5_single = era5_buffer.to_dataframe().reset_index().dropna()

In [ ]:
# extend the transect line
# reproject the transect to the UTM projection
gdf_utm = gpd.GeoDataFrame([transect], crs='EPSG:4326').to_crs(f'EPSG:{transect["utm_epsg"]}')

d = gdf_utm.to_crs('EPSG:4326').hvplot(
    geo=True, 
    tiles='EsriImagery', 
    color='red'
)

# define a function to extend the transect line 
def extend_line(geom: shapely.LineString, distance_m: float, ends="both"):
    coords = list(geom.coords)

    def extend_point(p1, p2, dist):
        p1, p2 = np.array(p1), np.array(p2)
        direction = (p2 - p1) / np.linalg.norm(p2 - p1)
        return tuple(p2 + direction * dist)

    new_coords = coords.copy()
    if ends in ("start", "both"):
        new_coords[0] = extend_point(coords[1], coords[0], distance_m)
    if ends in ("end", "both"):
        new_coords[-1] = extend_point(coords[-2], coords[-1], distance_m)

    return shapely.LineString(new_coords)

gdf_utm['geometry'] = gdf_utm['geometry'].apply(
    lambda geom: extend_line(geom, distance_m=100_000, ends="end")
)

In [ ]:
a * b * c * d

In [ ]:
# Set geometry information for ERA5 points  
era5_gdf = gpd.GeoDataFrame(
    era5_single, 
    geometry=[shapely.points([lon, lat]) for lon, lat in zip(era5_single.longitude, era5_single.latitude)], 
    crs='EPSG:4326'
)

# project to UTM projection, each transect has UTM zone information
original_crs = 'EPSG:4326'
utm_crs = int(gdf_utm['utm_epsg'].values)

era5_gdf = era5_gdf.to_crs(f'EPSG:{utm_crs}')

# calculate distance 
result = gpd.sjoin_nearest(era5_gdf, gdf_utm, distance_col='dist_m').sort_values('dist_m')
best_two = result.head(2)

In [ ]:
era5 = era5_buffer.hvplot.points(
    x='longitude',
    y='latitude',
    geo=True, 
    crs='EPSG:4326',
    tiles='EsriImagery',
    color='blue',
    hover_cols=['swh']
)

clipped_era5 = era5_gdf.hvplot(
    geo=True,
    tiles='EsriImagery', 
    crs=f'EPSG:{transect['utm_epsg']}', 
    color='red',
    width=600, 
    height=400
)

long_tr = gdf_utm.hvplot(
    geo=True, 
    tiles='EsriImagery', 
    crs=f'EPSG:{transect['utm_epsg']}', 
    width=600, 
    height=400
)

era5 * long_tr * clipped_era5

In [ ]:
# inspect the two closest points
first = ds.sel(latitude=8.5, longitude=81.5, time=slice('1980-01-01T00:00:00.000000000', '2020-01-01T00:00:00.000000000')).compute()
second = ds.sel(latitude=8.5, longitude=82, time=slice('1980-01-01T00:00:00.000000000', '2020-01-01T00:00:00.000000000')).compute()

first_graph = first.hvplot(
    x='time', 
    y='swh', 
    color='blue', 
    label='First, closer to the shore'
)

second_graph = second.hvplot(
    x='time', 
    y='swh', 
    color='red', 
    line_dash='dashed',
    label='Second, more offshore'
)

# first_graph * second_graph

## Select IPCC projection closest points

In [ ]:
# load the data lazily 
SCENARIO = 'ssp370'
DIR = f'../data/AR6_slr/total_{SCENARIO}_medium_confidence_rates.nc'

ds = xr.open_dataset(DIR)

In [ ]:
buffer_slr = 2

mask = (
    (ds.lon.values >= (LON_MIN-buffer_slr)) & (ds.lon.values <= (LON_MAX+buffer_slr)) &
    (ds.lat.values >= (LAT_MIN-buffer_slr)) & (ds.lat.values <= (LAT_MAX+buffer_slr))
)

print(f"Points in bbox: {mask.sum()} / {mask.size}")  # sanity checkdd

valid_idx = np.where(mask)[0]
ds_subset = ds.isel(locations=valid_idx)

# extend the transect 200 km 
transect_200 = gdf_utm.copy()

transect_200['geometry'] = transect_200['geometry'].apply(
    lambda geom: extend_line(geom, distance_m=200_000, ends="end")
)

In [ ]:
slr_clipped = ds_subset.sel(quantiles=0.5, years=2020).to_dataframe().reset_index()

slr_clipped_gdf = gpd.GeoDataFrame(
    slr_clipped, 
    geometry=[shapely.points(lon, lat) for lon, lat in zip(slr_clipped.lon, slr_clipped.lat)], 
    crs='EPSG:4326'
)

In [ ]:
# calculate distance
# project to UTM in transect 
slr_clipped_gdf = slr_clipped_gdf.to_crs(
    f'EPSG:{utm_crs}'
)

In [ ]:
clipped_slr = slr_clipped_gdf.to_crs('EPSG:4326').hvplot(
    geo=True,
    tiles='EsriImagery', 
    width=600, 
    height=400, 
    label='SLR', 
    color='green'
)

tr_200 = transect_200.to_crs('EPSG:4326').hvplot(
    geo=True, 
    tiles='EsriImagery', 
    width=600, 
    height=400, 
    label='Transect', 
)

# tr_200 * clipped_slr

In [ ]:
result = gpd.sjoin_nearest(slr_clipped_gdf, transect_200, distance_col='dist_m').sort_values('dist_m')
best_two = result.head(2)

print(result.head(3))

In [ ]:
from pcr import geo

idx_first = geo.find_closest(83, 9, ds.lon, ds.lat, 'locations')
point_first = ds.isel(locations=idx_first).sel(quantiles=0.5).compute()

idx_second = geo.find_closest(82, 9, ds.lon, ds.lat, 'locations')
point_second = ds.isel(locations=idx_second).sel(quantiles=0.5).compute()

idx_third = geo.find_closest(81, 9, ds.lon, ds.lat, 'locations')
point_third = ds.isel(locations=idx_third).sel(quantiles=0.5).compute()

In [ ]:
first_slr = point_first.hvplot(
    x="years", 
    y="sea_level_change_rate", 
    label='First, more offshore', 
) 

second_slr = point_second.hvplot(
    x="years", 
    y="sea_level_change_rate", 
    label='Second, closer to the coast',
)

third_slr = point_third.hvplot(
    x="years", 
    y="sea_level_change_rate", 
    label='Third, very close to the coast',
    # line_dash='dashed',
)

(first_slr * second_slr * third_slr).opts(legend_position='bottom_right', title=f'Scenario {SCENARIO}')

In [ ]:
tr_100 = gdf_utm.to_crs('EPSG:4326').hvplot(
    geo=True, 
    tiles='EsriImagery', 
    width=600, 
    height=400, 
    label='Transect', 
)

# tr_100 * clipped_slr

# Implementation in module 

In [ ]:
# select area using ipyleaflet map 
from ipyleaflet import Map, basemaps
from ipywidgets import Layout

from pcr.model import PCRModel
from pcr import builder, geo 

## Select Area of Interest

In [ ]:
# render map and zoom in to the area of interest
m = Map(
    basemap=basemaps.Esri.WorldImagery,
    scroll_wheel_zoom=True,
    center=(7.6031203577833315, 81.77331084939762),
    zoom=12,
    layout=Layout(width='800px', height='500px')
)

m

In [ ]:
# zoom in into B-I cell 
bbox_tr = [m.west, m.south, m.east, m.north]
bbox_tr

## Initialize model in this area

In [ ]:
# build wave model with no sea level rise for calibration 
# modelB1 = PCRModel(
#     # general config 
#     year_start = 2020, 
#     year_end = 2119, 
#     nr_simulation = 1000, 
#     nr_batch = 1000,
#     # erosion 
#     c1 = 1.209, 
#     c2 = 1.849
# )

# modelB3 = PCRModel(
#     # general config 
#     year_start = 2020, 
#     year_end = 2119, 
#     nr_simulation = 1000, 
#     nr_batch = 1000,
#     # erosion 
#     c1 = 1.339, 
#     c2 = 1.849
# )

modelT3 = PCRModel(
    # general config 
    year_start = 2020, 
    year_end = 2119, 
    nr_simulation = 1000, 
    nr_batch = 1000,
    # erosion 
    c1 = 2.046, 
    c2 = 0.800
)

In [ ]:
# build transect
modelT3.attach_transect(
    builder.build_transect(bbox_tr)
)

# build wave input 
hs, dir, tp, day, record_years, lon_wave, lat_wave = builder.build_wave_array(
    source='ARCO', 
    transect=modelT3.transect, 
    cds_api_key=cds_api_key, 
)
# attach to model 
modelT3.attach_wave_data(hs, dir, tp, day, record_years, lon_wave, lat_wave)

# calibrating the model to search rec_rate 
from pcr import calibration

# detect storm only once 
modelT3.detect_storms()
modelT3.rec_rate = calibration.get_or_calibrate_rec_rate(modelT3)

# simulate ssp126 
modelT3.nr_simulation = 50_000
scenario='ssp126'
rate_126, days126 = builder.build_slr_curve(scenario=scenario, transect=modelT3.transect, date_start=modelT3.date_start)
modelT3.attach_slr(rate_ar6=rate_126, days_ar6=days126, scenario=scenario)

modelT3.run_simulation()

In [ ]:
from pcr import visualisation
# import matplotlib.pyplot as plt

ax = visualisation.plot_exceedance(modelT3)

In [ ]:
modelT3

In [ ]:
modelT3.track_shoreline = None 
modelT3.track_time = None 

In [ ]:
import copy

modelT3_0 = copy.deepcopy(modelT3)

In [ ]:
modelT3_0.attach_slr([0, 0, 0], [0, 0, 0], scenario='0')

In [ ]:
modelT3_0.run_simulation()

In [ ]:
visualisation.plot_exceedance(modelT3_0)